In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 3 - Week 8 Bayesian Optimisation
# --------------------------------------------------
#
# - maximise the raw objective
# - use all observations through Week 7
# - fit an ARD Matern GP
# - optimise GP hyperparameters automatically
# - generate local + wide + global candidates
# - Expected Improvement is the primary acquisition
# - UCB and highest GP mean are diagnostics

In [2]:
X = np.load("function3/initial_inputs.npy")
Y = np.load("function3/initial_outputs.npy").reshape(-1)

assert len(X) == len(Y)
assert X.shape[1] == 3

print("X shape:", X.shape)
print("Y shape:", Y.shape)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:")
print(best_x)

print("\nCurrent best observed output:")
print(best_y)

print("\nY range:")
print("min =", np.min(Y))
print("max =", np.max(Y))
print("std =", np.std(Y))

X shape: (22, 3)
Y shape: (22,)

Current best observed input:
[0.364352 0.404312 0.451822]

Current best observed output:
-0.0149952064996286

Y range:
min = -0.3989255131463011
max = -0.0149952064996286
std = 0.07717378505484712


In [3]:
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(3) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
1.62**2 * Matern(length_scale=[0.753, 1.18, 0.238], nu=2.5) + WhiteKernel(noise_level=1e-08)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [4]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


ARD lengthscales:
[0.7530951  1.1801291  0.23835465]

Normalised inverse-lengthscale sensitivity:
[0.20843306 0.1330108  0.65855613]


In [5]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:", local_scale)
print("Wide widths:", wide_scale)


Local widths: [0.1        0.1        0.05958866]
Wide widths: [0.2        0.2        0.11917733]


In [6]:
rng = np.random.default_rng(42)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(50000, 3)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(30000, 3)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(50000, 3)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

print(
    "Candidates before filtering:",
    len(candidates)
)

Candidates before filtering: 130000


In [7]:
tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 129960


In [8]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

print("Predictions complete.")

Predictions complete.


In [9]:
def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [10]:
EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\nPRIMARY EI RESULT")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


PRIMARY EI RESULT
candidate = [3.46048275e-01 9.53892901e-01 6.59099526e-04]
mean = -0.014531330672314838
std = 0.05326600415123295
EI = 0.021482804883981566


In [11]:
y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\nEI sensitivity check:\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI sensitivity check:

xi = 0.000000e+00 
 candidate = [3.46048275e-01 9.53892901e-01 6.59099526e-04] 
 mean = -0.014531 
 std = 0.053266 
 EI = 0.0214828 

xi = 7.717379e-04 
 candidate = [3.46048275e-01 9.53892901e-01 6.59099526e-04] 
 mean = -0.014531 
 std = 0.053266 
 EI = 0.02109649 

xi = 3.858689e-03 
 candidate = [3.46048275e-01 9.53892901e-01 6.59099526e-04] 
 mean = -0.014531 
 std = 0.053266 
 EI = 0.0195958 

xi = 7.717379e-03 
 candidate = [3.46048275e-01 9.53892901e-01 6.59099526e-04] 
 mean = -0.014531 
 std = 0.053266 
 EI = 0.01782003 



In [12]:
print("\nUCB diagnostic:\n")

for beta in [
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB diagnostic:

beta=0.1 
 candidate = [4.95431873e-01 8.20834827e-01 8.05447355e-04] 
 mean = -0.008662 
 std = 0.03839 
 UCB = -0.004823 

beta=0.25 
 candidate = [4.95431873e-01 8.20834827e-01 8.05447355e-04] 
 mean = -0.008662 
 std = 0.03839 
 UCB = 0.000935 

beta=0.5 
 candidate = [4.13687443e-01 8.44242435e-01 5.40531971e-04] 
 mean = -0.010371 
 std = 0.045117 
 UCB = 0.012187 

beta=1.0 
 candidate = [0.18869213 0.98329467 0.00135883] 
 mean = -0.024799 
 std = 0.063534 
 UCB = 0.038735 



In [13]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.34559252 0.56247101 0.42432794]
mean = -0.007439945258327724
std = 0.009430422903237746


In [14]:
# --------------------------------------------------
# Function 3 - ARD trust-region search
# --------------------------------------------------

trust_half_width = np.clip(
    0.5 * lengthscales,
    0.025,
    0.15
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("Current best:", best_x)
print("ARD lengthscales:", lengthscales)
print("Trust-region half widths:", trust_half_width)
print("Lower bounds:", lower)
print("Upper bounds:", upper)

Current best: [0.364352 0.404312 0.451822]
ARD lengthscales: [0.7530951  1.1801291  0.23835465]
Trust-region half widths: [0.15       0.15       0.11917733]
Lower bounds: [0.214352   0.254312   0.33264467]
Upper bounds: [0.514352   0.554312   0.57099933]


In [15]:
rng = np.random.default_rng(42)

tr_candidates = rng.uniform(
    lower,
    upper,
    size=(150000, 3)
)

tree = cKDTree(X)

distance, _ = tree.query(
    tr_candidates,
    k=1
)

tr_candidates = tr_candidates[
    distance > 0.01
]

print(
    "Trust-region candidates:",
    len(tr_candidates)
)

tr_mu, tr_sigma = gp.predict(
    tr_candidates,
    return_std=True
)

Trust-region candidates: 149908


In [16]:
tr_EI = expected_improvement(
    tr_mu,
    tr_sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(tr_EI)

print("\nTrust-region EI:")
print("candidate =", tr_candidates[ei_idx])
print("mean =", tr_mu[ei_idx])
print("std =", tr_sigma[ei_idx])
print("EI =", tr_EI[ei_idx])


Trust-region EI:
candidate = [0.3199379  0.55160571 0.41614038]
mean = -0.007800488602077313
std = 0.010938155835327956
EI = 0.008872416148055458


In [17]:
mean_idx = np.argmax(tr_mu)

print("\nTrust-region highest predicted mean:")
print("candidate =", tr_candidates[mean_idx])
print("mean =", tr_mu[mean_idx])
print("std =", tr_sigma[mean_idx])


Trust-region highest predicted mean:
candidate = [0.34089796 0.55337783 0.42854175]
mean = -0.0074868878715688425
std = 0.008903921010493445


In [18]:
print("\nTrust-region UCB:\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = tr_mu + beta * tr_sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", tr_candidates[idx],
        "\n mean =", round(tr_mu[idx], 6),
        "\n std =", round(tr_sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


Trust-region UCB:

beta=0.1 
 candidate = [0.33234786 0.55398733 0.42064452] 
 mean = -0.007519 
 std = 0.010054 
 UCB = -0.006514 

beta=0.25 
 candidate = [0.33234786 0.55398733 0.42064452] 
 mean = -0.007519 
 std = 0.010054 
 UCB = -0.005006 

beta=0.5 
 candidate = [0.30966473 0.55220203 0.41719331] 
 mean = -0.008022 
 std = 0.011404 
 UCB = -0.00232 

beta=1.0 
 candidate = [0.28534523 0.55186824 0.41048877] 
 mean = -0.009277 
 std = 0.01308 
 UCB = 0.003804 



In [19]:
# --------------------------------------------------
# Final Function 3 Week 8 selection
# --------------------------------------------------
#
# UCB calibration inside the ARD trust region showed:
# - beta=0.1 and beta=0.25 selected the same candidate
# - higher beta values moved toward greater uncertainty
#   but lower predicted mean
#
# beta=0.25 is therefore used as a balanced
# exploration-exploitation setting.

beta = 0.25

UCB = tr_mu + beta * tr_sigma
final_idx = np.argmax(UCB)

week8_candidate = tr_candidates[final_idx]

print("Week 8 Function 3 candidate:")
print(week8_candidate)

print("\nPredicted mean:")
print(tr_mu[final_idx])

print("\nPredicted std:")
print(tr_sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week8_candidate
)

print("\nPortal format:")
print(portal)

Week 8 Function 3 candidate:
[0.33234786 0.55398733 0.42064452]

Predicted mean:
-0.007519341213799616

Predicted std:
0.01005382815183059

UCB:
-0.005005884175841968

Portal format:
0.332348-0.553987-0.420645
